# صوت الفيديو الترويجي بـ ElevenLabs
شغّل الخلايا بالترتيب.
1. **الخلية 2:** ضع مفتاح API في أسرار Colab (أيقونة المفتاح 🔑 على اليسار) باسم `ELEVEN_API_KEY`. **لا ترسل المفتاح لأحد.**
2. **الخلية 4:** تعرض الأصوات المتاحة في حسابك. انسخ `voice_id` للصوت الذي يعجبك في `VOICE_ID`.
3. **الخلية 4:** تعرض أيضًا النماذج، ويُختار **Eleven v4** تلقائيًا.
4. **الخلية 5:** تجربة جملة واحدة.
5. **الخلية 6:** الصوت كاملًا. كل جملة تُحفظ وحدها، فإن أعدت التشغيل لا تُولَّد إلا الجمل المذكورة في `REDO` أو التي لم تُولَّد بعد.
6. أرسل لي `promo2_voice.zip` كما هو.


In [ ]:
!pip -q install requests pydub "audioop-lts; python_version>='3.13'"


In [ ]:
from google.colab import userdata
API_KEY = userdata.get("ELEVEN_API_KEY")
MODEL = "eleven_v4"             # فارغ = يختار Eleven v4 تلقائيًا (الخلية 4 تعرض النماذج). أو ضع مثلًا "eleven_multilingual_v2"
VOICE_ID = "G1HOkzin3NMwRHSq60UI"          # ضع هنا voice_id من الخلية 4
SETTINGS = {"stability": 0.5, "similarity_boost": 0.8, "style": 0.3, "use_speaker_boost": True, "speed": 0.95}
GAP = 0.6              # صمت بين الجمل (ثوانٍ)
LEAD = 0.4
REDO = [7, 8]              # أرقام الجمل التي تريد إعادة توليدها (تبدأ من 1)، مثلًا [3, 7]
LINES = [
 "لكل أستاذ تعليم ثانوي، يدرّس في القسم أو في دروس الدعم، ولكل تلميذ يدرس في الثانوية، وخاصة تلاميذ البكالوريا.",
 "إليك مكتبة الأستاذ سوايسية هشام: أكثر من مئة وأربعين ملفًا، للسنة الأولى والثانية والثالثة ثانوي، مطابقة للمنهاج وللتوزيع السنوي الجديد.",
 "ملخصات، ومذكرات، وتمارين نموذجية مع الحلول، وتمارين البكالوريا مع الحلول، بثلاث صيغ: PDF ووورد ولاتاك، وكل الملفات قابلة للتعديل.",
 "في السنة الثالثة ثانوي، كل درس فيه أربعة ملفات، لجميع الشعب العلمية ولشعبة التسيير والاقتصاد.",
 "الدروس حسب التوزيع السنوي: الاشتقاقية والاستمرارية، الدالة الأسية، الدالة اللوغاريتمية، النهايات، التزايد المقارن، المتتاليات العددية، الدوال الأصلية والحساب التكاملي، الأعداد والحساب، الاحتمالات، الأعداد المركبة، الهندسة في الفضاء.",
 "شعبة التسيير والاقتصاد: المتتاليات العددية، الاشتقاقية والاستمرارية، النهايات ودراسة الدوال، الدوال الأصلية والحساب التكاملي، الدالة اللوغاريتمية، الدالة الأسية، الإحصاء، الاحتمالات.",
 "السنة الثانية ثانوي، الشعب العلمية: الدوال، الاشتقاقية، الاحتمالات، المرجح، النهايات، الزوايا الموجهة وحساب المثلثات، التحويلات النقطية، الجداء السلمي، المتتاليات العددية، الهندسة في الفضاء.",
 "السنة الأولى ثانوي، جذع مشترك علوم وتكنولوجيا: الأعداد والحساب، عموميات على الدوال، الحساب الشعاعي، الدوال المرجعية، العبارات الجبرية، الهندسة المستوية، الهندسة في الفضاء، الإحصاء.",
 "لمعرفة كيفية اقتناء جميع الملفات، أرسل رسالة على فايسبوك أو إنستغرام، أو على واتساب عبر هذا الرقم."
]
# تصحيح النطق: الكلمة كما تظهر على الشاشة -> الكتابة التي تُنطق (تشكيل أو كتابة صوتية).
# كلمة واحدة مقابل كلمة واحدة حتى يبقى توقيت الكتابة صحيحًا.
FIX = {
 "السنة": "السَّنَة",
 "الجداء": "الجُدَاء",
 "السلمي": "السُّلَّمِي",
 "الموجهة": "المُوَجَّهَة",
 "الشعاعي": "الشُّعَاعِي",
 "المرجعية": "المَرْجِعِيَّة",
 "عموميات": "عُمُومِيَّات",
 "التكاملي": "التَّكَامُلِي",
 "المقارن": "المُقَارَن",
 "المستوية": "المُسْتَوِيَة",
 "المثلثات": "المُثَلَّثَات",
 "التزايد": "التَّزَايُد",
 "الاستمرارية": "الاِسْتِمْرَارِيَّة",
 "أستاذ": "أُسْتَاذِ",
 "تعليم": "تَعْلِيمٍ",
 "يدرّس": "يُدَرِّسُ",
 "يدرس": "يَدْرُسُ",
 "إليك": "إِلَيْكَ",
 "الشعب": "الشُّعَب",
 "ولشعبة": "وَلِشُعْبَة",
 "شعبة": "شُعْبَة",
 "الأسية": "الأُسِّيَّة",
 "صيغ": "صِيَغٍ",
 "ملفًا": "مِلَفًّا",
 "ملفات": "مِلَفَّات",
 "الملفات": "المِلَفَّات",
 "والملف": "وَالمِلَفُّ",
 "ولاتاك": "وLaTeX",
 "ووورد": "وَوُورْدْ",
 "ملخصات": "مُلَخَّصَات",
 "ومذكرات": "وَمُذَكِّرَات",
 "مذكرة": "مُذَكِّرَة",
 "وملخص": "وَمُلَخَّص",
 "ملخص": "مُلَخَّص",
 "المرجح": "المُرَجِّح",
 "المركبة": "المُرَكَّبَة",
 "المتتاليات": "المُتَتَالِيَات",
 "اللوغاريتمية": "اللُّوغَارِيتْمِيَّة",
 "الاشتقاقية": "الاِشْتِقَاقِيَّة",
 "والاستمرارية": "وَالاِسْتِمْرَارِيَّة",
 "جذع": "جِذْع",
 "اقتناء": "اقْتِنَاء",
 "فايسبوك": "فَايْسْبُوكْ",
 "إنستغرام": "إِنْسْتَغْرَامْ",
 "واتساب": "وَاتْسَابْ",
 "ثانوي": "ثَانَوِي",
}
def spoken(line):
    return " ".join(FIX.get(w.strip("،:؟.!"), w.strip("،:؟.!")) + w[len(w.rstrip("،:؟.!")):] if w.strip("،:؟.!") in FIX else w for w in line.split())


In [ ]:
import requests
print("المفتاح موجود:", bool(API_KEY))
r = requests.get("https://api.elevenlabs.io/v1/voices", headers={"xi-api-key": API_KEY})
print("الحالة:", r.status_code)
j = r.json()
if "voices" not in j:
    print(j)
for v in j.get("voices", []):
    print(v["voice_id"], "|", v["name"], "|", v.get("category"), "|", v.get("labels", {}))

# النماذج المتاحة لحسابك: نختار Eleven v4 إن لم تحدد MODEL
m = requests.get("https://api.elevenlabs.io/v1/models", headers={"xi-api-key": API_KEY}).json()
print("\nالنماذج:")
for x in m if isinstance(m, list) else []:
    print(" ", x.get("model_id"), "|", x.get("name"), "| tts:", x.get("can_do_text_to_speech"))
if not MODEL:
    v4 = [x["model_id"] for x in (m if isinstance(m, list) else []) if x.get("can_do_text_to_speech") and ("v4" in x.get("model_id", "").lower() or "v4" in x.get("name", "").lower())]
    MODEL = v4[0] if v4 else "eleven_multilingual_v2"
print("\nالنموذج المستعمل:", MODEL)


In [ ]:
import base64, io, os, json
from pydub import AudioSegment
from IPython.display import Audio, display
os.makedirs("parts", exist_ok=True)

def _post(url, text, settings):
    return requests.post(url, headers={"xi-api-key": API_KEY}, json={"text": text, "model_id": MODEL, "voice_settings": settings})

def tts(text):
    base = f"https://api.elevenlabs.io/v1/text-to-speech/{VOICE_ID}"
    simple = {k: SETTINGS[k] for k in ("stability", "similarity_boost") if k in SETTINGS}
    r = None
    for url in (base + "/with-timestamps?output_format=mp3_44100_128", base + "?output_format=mp3_44100_128"):
        for st in (SETTINGS, simple, {"stability": 0.5}):
            r = _post(url, text, st)
            if r.ok: break
        if r.ok: break
    if not r.ok: print(r.status_code, r.text[:500]); r.raise_for_status()
    if r.headers.get("content-type", "").startswith("application/json"):
        j = r.json(); seg = AudioSegment.from_file(io.BytesIO(base64.b64decode(j["audio_base64"])), format="mp3")
        a = j.get("alignment") or j.get("normalized_alignment")
    else:                                   # بدون توقيت: نقدّر توقيت الكلمات من طولها
        seg = AudioSegment.from_file(io.BytesIO(r.content), format="mp3"); a = None
    words, cur = [], None
    if a:
        for ch, s, e in zip(a["characters"], a["character_start_times_seconds"], a["character_end_times_seconds"]):
            if ch.isspace():
                if cur: words.append(cur); cur = None
            elif cur is None: cur = {"text": ch, "start": s, "end": e}
            else: cur["text"] += ch; cur["end"] = e
        if cur: words.append(cur)
    else:
        ws = text.split(); tot = sum(len(x) for x in ws); t = 0.15; dur = len(seg) / 1000 - 0.3
        for x in ws:
            dt = dur * len(x) / tot; words.append({"text": x, "start": t, "end": t + dt}); t += dt
    return seg, words

seg, w = tts(spoken(LINES[0]))
print(len(w), "كلمة"); display(Audio(seg.export(format="mp3").read()))


In [ ]:
import zipfile
from google.colab import files
full = AudioSegment.silent(duration=int(LEAD * 1000)); out = []
for i, line in enumerate(LINES, 1):
    mp3, js = f"parts/{i:02d}.mp3", f"parts/{i:02d}.json"
    if i in REDO or not os.path.exists(mp3):
        seg, words = tts(spoken(line)); seg.export(mp3, format="mp3", bitrate="192k")
        json.dump(words, open(js, "w"), ensure_ascii=False)
        print(i, "توليد", round(len(seg) / 1000, 1), "ث")
    seg = AudioSegment.from_file(mp3); words = json.load(open(js))
    shown = line.split()
    if len(shown) == len(words):            # نعرض الكلمات كما في السيناريو (قبل تصحيح النطق)
        for w_, s_ in zip(words, shown): w_["text"] = s_
    t0 = len(full) / 1000
    out.append({"i": i, "text": line, "start": t0, "end": t0 + len(seg) / 1000,
                "words": [{**w_, "start": w_["start"] + t0, "end": w_["end"] + t0} for w_ in words]})
    full += seg + AudioSegment.silent(duration=int(GAP * 1000))
full.export("voice.mp3", format="mp3", bitrate="192k")
json.dump({"voice": VOICE_ID, "model": MODEL, "duration": len(full) / 1000, "lines": out}, open("timings.json", "w"), ensure_ascii=False, indent=1)
with zipfile.ZipFile("promo2_voice.zip", "w") as z:
    z.write("voice.mp3"); z.write("timings.json")
print("المدة:", round(len(full) / 1000), "ثانية"); display(Audio("voice.mp3"))
files.download("promo2_voice.zip")
